In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import nivapy3 as nivapy
import pandas as pd
import teotil3 as teo

In [2]:
# Connect to JupyterHub's PostGIS database
eng = nivapy.da.connect_postgis()

# Define datasets of interest
# Year for admin. boundaries
admin_year = 2024
reg_gdf = teo.io.get_regine_geodataframe(eng, admin_year)

# Determine hydrological connectivity
reg_gdf = teo.io.assign_regine_hierarchy(
    reg_gdf,
    regine_col="regine",
    regine_down_col="regine_down",
    order_coastal=False,
    nan_to_vass=True,
    land_to_vass=True,
    add_offshore=True,
)

# Build network graph from adjacency matrix
g = teo.model.build_graph(reg_gdf, id_col="regine", next_down_col="regine_down")

Connection successful.
100.00 % of regines assigned.


In [3]:
reg_id = "002.CB0"
g2 = nx.dfs_tree(g.reverse(), reg_id).reverse()
reg_list = list(g2.nodes)
reg_list

['002.CB0',
 '002.CBA0',
 '002.CBAA',
 '002.CBAB0',
 '002.CBABA',
 '002.CBABB',
 '002.CBABC',
 '002.CBAC',
 '002.CBB1',
 '002.CBB2',
 '002.CBB3',
 '002.CBB4',
 '002.CBB5',
 '002.CBB6',
 '002.CBC',
 '002.CC0',
 '002.CCA',
 '002.CCB',
 '002.CD11',
 '002.CD120',
 '002.CD12A',
 '002.CD12B',
 '002.CD1A',
 '002.CD1B',
 '002.CD1C',
 '002.CD1D',
 '002.CD2',
 '002.CDA0',
 '002.CDAA',
 '002.CDAB',
 '002.CDB',
 '002.CDC',
 '002.CDD',
 '002.CE',
 '002.CF1',
 '002.CF21',
 '002.CF22',
 '002.CF2A',
 '002.CF2B1',
 '002.CF2B21',
 '002.CF2B22',
 '002.CF2B2A',
 '002.CF2B2B',
 '002.CF2C1',
 '002.CF2C2',
 '002.CF2C3',
 '002.CF2C41',
 '002.CF2C42',
 '002.CF2C4A',
 '002.CF2C4B',
 '002.CF2D',
 '002.CFZ',
 '002.CG',
 '002.CH']

In [5]:
xl_path = r"/home/jovyan/projects/oslofjord_modelling/oslomod_phase3_teotil/data/wwtp_scenarios_summary.xlsx"
df = pd.read_excel(xl_path)

df = df.query("regine in @reg_list")
df.head()

,scenario,anlegg_nr,kilderefnr,anlegg_name,martini_name,martini_river_or_internal,regine,year,activity,kommune,...,bof5_in_tonnes,bof5_out_tonnes,kof_in_tonnes,kof_out_tonnes,ss_in_tonnes,ss_out_tonnes,totn_in_tonnes,totn_out_tonnes,totp_in_tonnes,totp_out_tonnes
131,Baseline,3205.0078.01,0231AL18,NRA (Nedre Romerike avløpsanlegg),NaN,NaN,002.CB0,2017,Avløpsnett og -rensing,Lillestrøm,...,3419.271,223.429,8315.073,810.474,2763.503,193.445,546.691,153.074,76.731,6.692
132,Baseline,3205.0078.01,0231AL18,NRA (Nedre Romerike avløpsanlegg),NaN,NaN,002.CB0,2018,Avløpsnett og -rensing,Lillestrøm,...,3317.243,187.268,7163.563,568.503,2846.183,199.233,605.224,168.643,78.240,6.410
133,Baseline,3205.0078.01,0231AL18,NRA (Nedre Romerike avløpsanlegg),NaN,NaN,002.CB0,2019,Avløpsnett og -rensing,Lillestrøm,...,3909.622,572.455,9465.656,1535.205,2909.726,203.681,713.824,321.121,96.225,14.284
215,Baseline,3232.0035.01,0233AL02,Rotnes renseanlegg,NaN,NaN,002.CD11,2017,Avløpsnett og -rensing,Nittedal,...,158.606,27.843,379.601,74.460,247.546,24.372,32.850,26.937,5.013,0.611
216,Baseline,3232.0035.01,0233AL02,Rotnes renseanlegg,NaN,NaN,002.CD11,2018,Avløpsnett og -rensing,Nittedal,...,160.183,38.113,370.196,78.064,220.852,24.591,32.850,26.937,5.018,0.625


In [12]:
par = "totn"
for site_id, site_df in df.groupby(["anlegg_nr"]):
    name = site_df.iloc[0]["anlegg_name"]
    sc_df = site_df.groupby("scenario").sum()[
        [f"{par}_in_tonnes", f"{par}_out_tonnes"]
    ] / 3
    for scen in ["Scenario_A", "Scenario_B"]:
        delta = sc_df.loc['Baseline', f"{par}_out_tonnes"] - sc_df.loc[scen, f"{par}_out_tonnes"]
        print(name, scen, delta, 'tonnes')

NRA (Nedre Romerike avløpsanlegg) Scenario_A 77.45866666666666 tonnes
NRA (Nedre Romerike avløpsanlegg) Scenario_B 83.67766666666665 tonnes
Rotnes renseanlegg Scenario_A 19.71 tonnes
Rotnes renseanlegg Scenario_B 20.039 tonnes
Slattum renseanlegg (Nedlagt) Scenario_A 18.759 tonnes
Slattum renseanlegg (Nedlagt) Scenario_B 19.071 tonnes
Åneby renseanlegg Scenario_A 4.730666666666668 tonnes
Åneby renseanlegg Scenario_B 12.851333333333333 tonnes
Harestua renseanlegg Scenario_A 0.0 tonnes
Harestua renseanlegg Scenario_B 0.16600000000000215 tonnes


In [16]:
par = "totn"
for scen in ["Scenario_A", "Scenario_B"]:
    sc_df = df.groupby("scenario").sum()[
        [f"{par}_in_tonnes", f"{par}_out_tonnes"]
    ] / 3
    delta = sc_df.loc['Baseline', f"{par}_out_tonnes"] - sc_df.loc[scen, f"{par}_out_tonnes"]
    print(scen, delta, 'tonnes')

Scenario_A 120.65833333333336 tonnes
Scenario_B 135.80500000000004 tonnes


In [14]:
sc_df

,totn_in_tonnes,totn_out_tonnes
scenario,,
Baseline,16.637667,12.810667
Scenario_A,16.637667,12.810667
Scenario_B,16.637667,12.644667
